# Step 3 – Energy system optimisation (oemof.solph)
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Model the **five heat producers** of Step 2 in oemof.solph and optimise their **capacities (investment) and hourly dispatch** to meet the **aggregated heat demand** of the ten consumers. **No thermal storage.** |
| **Reads** | `data/step2/twn_data/consumers.csv` (which profiles) · demand profiles · `data/input_data_DH1.csv` (placeholder prices) |
| **Writes** | `data/step3/` (oemof input time series + technology/carrier tables, reused by Step 4–5) · `results/step3/` |
| **Grading** | C – formulate the dispatch problem (variables + units, objective, capacities, demand balance, parameters), report solution status, verify demand is met, analyse dispatch and periods at capacity |
| **Tutorial reference** | `OEMOF Tutorial/oemof.ipynb` → *district_heating_supply_1/2* |

Legend: 🔧 ASSUMPTION · ✏️ TODO

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import oemof.solph as solph
from oemof.solph import processing

from helpers import (DATA, RESULTS, SOLVER, COUNTRY, LATITUDE, YEAR, INTEREST_RATE, LCOH, epc,
                     load_demand_profiles, build_and_solve_heat_system, summarize_units,
                     heat_dispatch, plot_dispatch, UNIT_COLORS)

IN = DATA / "step3"
OUT = RESULTS / "step3"
IN.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Aggregated heat demand of the ten Step 2 consumers
The profile names are read from the Step 2 network (`consumers.csv`, column `profile`), so Step 2 and Step 3 always use the same consumers.

**Unit conversion:** demand profiles are in kW (hourly) → divide by 1000 → **MW**. With hourly time steps, 1 MW for one hour = 1 MWh, so all oemof energy results are in MWh and costs in €/MWh.

In [ ]:
consumers = pd.read_csv(DATA / "step2" / "twn_data" / "consumers.csv")
STEP2_LOADS = list(consumers["profile"])
demand = load_demand_profiles()                                  # kW

heat_demand_MW = demand[STEP2_LOADS].sum(axis=1) / 1000          # MW
print("Profiles:", STEP2_LOADS)
print(f"Peak {heat_demand_MW.max():.3f} MW | mean {heat_demand_MW.mean():.3f} MW | "
      f"annual {heat_demand_MW.sum():.0f} MWh | full-load hours {heat_demand_MW.sum()/heat_demand_MW.max():.0f} h")

## 2. Price time series (placeholder)
The oemof tutorial file `input_data_DH1.csv` (hourly 2019: `gas price`, `el_spot_price` in €/MWh) is used as a **placeholder** so that the model runs.

✏️ **TODO:** replace with data for the chosen country/year, e.g. day-ahead electricity prices for the Swedish bidding zone (Nord Pool / ENTSO-E Transparency) and a gas price (e.g. TTF), keeping 8760 hourly rows and the column names. Consider whether grid fees/taxes should be added to the electricity price.

In [ ]:
prices = pd.read_csv(DATA / "input_data_DH1.csv", sep=";", index_col=0, parse_dates=True)
assert len(prices) == len(heat_demand_MW) == 8760, "Price and demand series must both be 8760 hourly values"
prices[["gas price", "el_spot_price"]].describe().round(1)

## 3. Solar irradiance profile (own profile, as required for solar collectors)
A simple, transparent **synthetic** profile for the chosen location (latitude in `helpers.py`):
1. Solar elevation from declination and hour angle (standard solar geometry).
2. Clear-sky horizontal irradiance ≈ 1000 W/m² · sin(elevation).
3. × monthly **clearness index** (cloudiness) 🔧.
4. Collector capacity factor `solar_cf` = irradiance / 1000 W/m² → output = installed peak capacity [MW] × `solar_cf`.

🔧 / ✏️ Justify the clearness indices, or (better) replace the profile with hourly irradiance from **PVGIS** (EU JRC) for your location and year. Collector efficiency/tilt are neglected – state this as a limitation.

In [ ]:
# 🔧 ASSUMPTION - monthly clearness index for the chosen location (placeholder)
CLEARNESS = {1: 0.25, 2: 0.35, 3: 0.45, 4: 0.50, 5: 0.55, 6: 0.55,
             7: 0.55, 8: 0.50, 9: 0.45, 10: 0.35, 11: 0.25, 12: 0.20}

t = prices.index
doy, hour = t.dayofyear.values, t.hour.values + 0.5                     # hour centre
decl = np.radians(23.45) * np.sin(np.radians(360 * (284 + doy) / 365))   # declination
hour_angle = np.radians(15 * (hour - 12))
lat = np.radians(LATITUDE)
sin_elev = np.clip(np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.cos(hour_angle), 0, None)
irradiance = 1000 * sin_elev * t.month.map(CLEARNESS).values             # W/m2
solar_cf = pd.Series(np.clip(irradiance / 1000, 0, 1), index=t)

print(f"Solar full-load hours: {solar_cf.sum():.0f} h/a")
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
solar_cf.resample("ME").mean().plot.bar(ax=ax[0], title="Monthly mean capacity factor", color="#FBC02D")
solar_cf["2019-06-20":"2019-06-23"].plot(ax=ax[1], title="Example: 3 summer days")
plt.tight_layout(); plt.show()

## 4. Assemble and save the oemof input table
One table with everything time-dependent, hourly, in MW and €/MWh. Saved to `data/step3/oemof_input.csv` → Step 4 adds the network losses to it.

In [ ]:
data = pd.DataFrame({
    "heat demand":   heat_demand_MW.values,           # MW
    "gas price":     prices["gas price"].values,      # €/MWh fuel
    "el_spot_price": prices["el_spot_price"].values,  # €/MWh electricity
    "solar_cf":      solar_cf.values,                 # -  (0..1)
}, index=prices.index)
data.index.name = "date"
data.to_csv(IN / "oemof_input.csv")
data.describe().round(3)

## 5. Technology and energy-carrier parameters
The project requires, for each technology: **efficiency/COP, variable operating cost, fuel/input cost and the source of each parameter.**
Placeholder values are taken from the oemof tutorial where possible; ✏️ replace/verify them with the **Danish Energy Agency – Technology Data for Generation of Electricity and District Heating** (link in the project description) or other cited sources.

**Technologies** (`tech`, one row per producer, same order/ids as Step 2: P0…P4)

| column | unit | meaning |
|---|---|---|
| `input` | – | energy carrier consumed (row in `carriers`) |
| `efficiency` | – | heat out / input in (COP for the heat pump; 1 for solar, its yield is in `solar_cf`) |
| `capex_eur_per_MW` | €/MW_heat | specific investment → annualised with `epc()` |
| `var_om_eur_per_MWh` | €/MWh_heat | variable O&M |
| `max_capacity_MW` | MW | upper bound for the investment (NaN = unlimited) |
| `lifetime_yr` | a | economic lifetime |

**Energy carriers** (`carriers`): price (time series for gas/electricity from section 2) and CO₂ emission factor per MWh of *input* (used in Step 5; already reported here).

In [ ]:
# 🔧 ASSUMPTIONS - technology data (placeholders; fill in sources!)
tech = pd.DataFrame([
    # unit               input          eff   capex €/MW  varO&M €/MWh  max MW  life  source (✏️)
    ["biomass boiler",  "biomass",      0.90, 700_000,    3.0,          np.nan, 25, "✏️ DEA technology data – wood chip boiler"],
    ["gas boiler",      "gas",          0.95, 60_000,     1.10,         np.nan, 25, "oemof tutorial (DH supply 2) / ✏️ DEA"],
    ["heat pump",       "electricity",  3.5,  500_000,    1.2,          0.6,    25, "oemof tutorial; max = 🔧 heat-source limit"],
    ["electric boiler", "electricity",  0.99, 70_000,     1.0,          np.nan, 20, "oemof tutorial (eff) / ✏️ DEA (cost)"],
    ["solar thermal",   "solar",        1.0,  450_000,    0.5,          0.5,    25, "✏️ DEA – large solar DH; max = 🔧 land area"],
], columns=["unit", "input", "efficiency", "capex_eur_per_MW", "var_om_eur_per_MWh",
            "max_capacity_MW", "lifetime_yr", "source"]).set_index("unit")
tech["DHNx producer"] = ["P0", "P1", "P2", "P3", "P4"]

# 🔧 ASSUMPTIONS - energy carriers
carriers = pd.DataFrame([
    # carrier        price €/MWh   tCO2/MWh_in  source (✏️)
    ["gas",          np.nan,       0.202,       "time series (section 2); EF: IPCC 2006 / oemof tutorial (0.000202 t/kWh)"],
    ["biomass",      25.0,         0.0,         "✏️ wood-chip price; EF 0 = biogenic CO2 (state this convention)"],
    ["electricity",  np.nan,       0.04,        "time series (section 2); EF ✏️ Swedish grid average"],
    ["solar",        0.0,          0.0,         "free"],
], columns=["carrier", "price_eur_per_MWh", "ef_tCO2_per_MWh", "source"]).set_index("carrier")

# Derived numbers that must be reported per technology
report = tech[["DHNx producer", "input", "efficiency", "var_om_eur_per_MWh", "capex_eur_per_MW", "max_capacity_MW"]].copy()
report["mean input price [EUR/MWh_in]"] = [carriers.loc["biomass", "price_eur_per_MWh"], data["gas price"].mean(),
                                          data["el_spot_price"].mean(), data["el_spot_price"].mean(), 0.0]
report["mean fuel cost [EUR/MWh_heat]"] = report["mean input price [EUR/MWh_in]"] / tech["efficiency"]
report["CO2 [t/MWh_heat]"] = tech["input"].map(carriers["ef_tCO2_per_MWh"]) / tech["efficiency"]
report["annualised capex [EUR/MW/a]"] = [epc(c, i=INTEREST_RATE, n=n) for c, n in zip(tech["capex_eur_per_MW"], tech["lifetime_yr"])]

tech.to_csv(IN / "technology_parameters.csv")
carriers.to_csv(IN / "carriers.csv")
report.round(3)

## 6. The optimisation problem (write this down in the report)

```
 gas source ──► gas bus ──────► gas boiler ─────────┐
 biomass source ► biomass bus ► biomass boiler ─────┤
 electricity source ► el. bus ┬► heat pump ◄─ waste heat bus ◄─ waste heat source
                              └► electric boiler ───┤
 solar thermal (Source, profile) ───────────────────┤
                                                    ▼
                                        heat network bus ──► heat sink (demand)
```

**Sets:** units $u \in$ {biomass boiler, gas boiler, heat pump, electric boiler, solar thermal}, hours $t = 1..8760$ ($\Delta t$ = 1 h).

**Decision variables**
- $P_u \ge 0$ [MW] – installed heat capacity of unit $u$ (oemof `Investment`)
- $\dot Q_{u,t} \ge 0$ [MW] – heat output of unit $u$ in hour $t$ (oemof `Flow`)
- $\dot F_{u,t} \ge 0$ [MW] – fuel / electricity input of unit $u$

**Objective** – total annual system cost [€/a]
$$\min \sum_u \text{epc}(c^{inv}_u, i, n_u)\,P_u + \sum_t \sum_u \big( p^{fuel}_{u,t}\,\dot F_{u,t} + c^{O\&M}_u\,\dot Q_{u,t} \big)$$

**Constraints**
- Demand balance (heat bus), every hour: $\sum_u \dot Q_{u,t} = D_t$
- Conversion: $\dot Q_{u,t} = \eta_u\,\dot F_{u,t}$ (heat pump: electricity $= \dot Q/\text{COP}$, waste heat $= \dot Q\,(\text{COP}-1)/\text{COP}$)
- Capacity: $\dot Q_{u,t} \le P_u$; solar: $\dot Q_{solar,t} \le P_{solar}\,cf_t$
- Investment limits: $P_u \le P^{max}_u$
- No storage, no imports of heat → demand must be met by the five units every hour.

### 6.1 Build and solve
The model is built by `build_and_solve_heat_system()` in **`helpers.py`** (section 5) – written like the tutorial (`Bus`, `Source`, `Converter`, `Sink`, `Flow`, `Investment(ep_costs=epc(...))`, `Model`, `processing.results`).
It lives in `helpers.py` so that **Step 3, 4 and 5 solve exactly the same model**; open it next to this notebook to see every component.

In [ ]:
model, results, parts = build_and_solve_heat_system(data, tech, carriers, demand_col="heat demand", i=INTEREST_RATE)

sr = model.solver_results
status = sr["termination_condition"] if isinstance(sr, dict) else sr.Solver.Termination_condition
print("Solver:", SOLVER, "| termination condition:", status)
print(f"Objective (total annual cost): {model.objective():,.0f} EUR/a")

## 7. Verify that the heat demand is satisfied
Hourly heat balance: sum of all unit outputs must equal the demand in **every** hour.

In [ ]:
dispatch = heat_dispatch(results, parts)          # MW per unit + demand
units = [c for c in dispatch.columns if c != "demand"]
imbalance = dispatch[units].sum(axis=1) - data["heat demand"]
print(f"Max hourly imbalance: {imbalance.abs().max():.2e} MW")
print(f"Heat produced {dispatch[units].sum().sum():,.1f} MWh vs demand {data['heat demand'].sum():,.1f} MWh")
assert imbalance.abs().max() < 1e-4, "Demand not met!"
print("✅ Demand met in all 8760 hours")

## 8. Results per unit
Capacity, heat share, full-load hours, hours at maximum capacity, and the cost breakdown. The total must equal the objective (consistency check).

In [ ]:
summary = summarize_units(results, parts, tech, carriers)
print(f"Objective {model.objective():,.0f}  vs  sum of unit costs {summary.loc['TOTAL', 'total cost [EUR]']:,.0f}")
summary.round(1)

## 9. Dispatch over the year
Stacked heat production (daily means) and duration curves. ✏️ Describe which units supply base, mid and peak load and why (merit order = marginal cost; investment cost vs. full-load hours).

In [ ]:
plot_dispatch(dispatch, "Heat production by unit – Step 3")
plt.savefig(OUT / "dispatch.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
week = dispatch["2019-01-14":"2019-01-20"]
ax.stackplot(week.index, *[week[u] for u in units], labels=units, colors=[UNIT_COLORS[u] for u in units])
ax.plot(week.index, week["demand"], "k", lw=1, label="demand")
ax.set_title("Hourly dispatch, one winter week"); ax.set_ylabel("MW"); ax.legend(ncol=6, loc="upper center")
plt.tight_layout(); plt.show()

## 10. When do units reach their maximum capacity?
Count of hours per month in which each unit runs at ≥ 99.9 % of its installed capacity, and these hours marked on the demand curve – relate them to the demand profile (cold periods? solar hours?).

In [ ]:
caps = summary.loc[units, "capacity [MW]"]
at_cap = pd.DataFrame({u: dispatch[u] >= 0.999 * caps[u] for u in units if caps[u] > 1e-6})
hours_at_cap_by_month = at_cap.groupby(at_cap.index.month).sum()
hours_at_cap_by_month.index.name = "month"
display(hours_at_cap_by_month)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dispatch.index, dispatch["demand"], color="lightgrey", lw=0.6, label="demand")
for u in at_cap.columns:
    hrs = at_cap.index[at_cap[u]]
    ax.scatter(hrs, dispatch.loc[hrs, "demand"], s=3, color=UNIT_COLORS[u], label=f"{u} at max")
ax.set_ylabel("Demand [MW]"); ax.set_title("Hours in which a unit is at its maximum capacity"); ax.legend(markerscale=4, ncol=3)
plt.tight_layout(); plt.savefig(OUT / "hours_at_capacity.png", dpi=150); plt.show()

## 11. Operating cost per unit – which unit contributes most to system cost?

In [ ]:
cost_cols = ["annualised capex [EUR]", "fuel/electricity cost [EUR]", "var O&M cost [EUR]"]
summary.loc[units, cost_cols].plot.bar(stacked=True, figsize=(9, 4), title="Annual cost per unit – Step 3")
plt.ylabel("EUR/a"); plt.tight_layout(); plt.savefig(OUT / "costs_per_unit.png", dpi=150); plt.show()

# System levelised cost of heat with the tutorial LCOH() helper
invest_cost = (caps * tech["capex_eur_per_MW"]).sum()                   # EUR (overnight)
operation_cost = summary.loc["TOTAL", ["fuel/electricity cost [EUR]", "var O&M cost [EUR]"]].sum()
print(f"LCOH: {LCOH(invest_cost, operation_cost, data['heat demand'].sum(), i=INTEREST_RATE):.2f} EUR/MWh")

## 12. Save results (used by Step 4 and 5)

In [ ]:
summary.to_csv(OUT / "unit_summary.csv")
dispatch.to_csv(OUT / "dispatch.csv")
hours_at_cap_by_month.to_csv(OUT / "hours_at_capacity_by_month.csv")
print("Saved to", OUT)

## 13. Questions to answer
✏️

1. **Which generating units supply the majority of the heat demand? How is the load distributed among the units?** *(section 8 share of heat, section 9 plots)*
2. **Are there periods when any unit reaches its maximum defined capacity? Which units, and when?** *(section 10 – relate to demand peaks / solar availability / the `max_capacity_MW` limits)*
3. **What are the total operational costs for each unit, and which unit contributes most to system costs?** *(section 11)*
4. Report the formulation (section 6), the solution status (6.1) and the demand check (7), and justify every 🔧 parameter with a source.